<a href="https://colab.research.google.com/github/luyov-belli/cv_experience_ocr_bert/blob/main/notebooks/colab_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# cv_experience_ocr_bert: etapas pesadas en Colab

Este notebook **no contiene lógica propia**: clona el repo y ejecuta los scripts de `src/`. Todo cambio de código se hace en la laptop, va a una rama y entra a `main` por PR. Aquí solo se hace `git pull`.

Las salidas pesadas (imágenes, OCR, pesos) se guardan en Google Drive mediante enlaces simbólicos, porque la máquina de Colab se borra al cerrar la sesión.

**Antes de empezar:**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4* (la descarga no la necesita, pero el OCR y BERT sí).
2. Panel izquierdo → 🔑 *Secrets* → crear `HF_TOKEN` con tu token de Hugging Face y activar *Notebook access*.

## 0. Preparación (correr siempre al abrir el notebook)

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'Sin GPU'

from google.colab import drive
drive.mount('/content/drive')

name, memory.total [MiB]
Tesla T4, 15360 MiB
Mounted at /content/drive


In [2]:
import os

REPO_URL = 'https://github.com/luyov-belli/cv_experience_ocr_bert'
REPO = '/content/cv_experience_ocr_bert'
DRIVE = '/content/drive/MyDrive/cv_experience_ocr_bert'
BRANCH = 'main'  # para probar una rama antes de fusionarla, cambia esto

if not os.path.exists(REPO):
    !git clone -q {REPO_URL} {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!git log --oneline -1

# Carpetas pesadas -> Drive (persisten entre sesiones)
for sub in ['data/raw', 'data/ocr', 'models/bert']:
    target = f'{DRIVE}/{sub}'
    os.makedirs(target, exist_ok=True)
    if not os.path.islink(sub):
        !rm -rf {sub}
        os.symlink(target, sub)
!ls -la data models

/content/cv_experience_ocr_bert
b889996 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #4 from luyov-belli/fix/colab-pyarrow
data:
total 16
drwxr-xr-x 4 root root 4096 Oct 26 03:33 .
drwxr-xr-x 9 root root 4096 Oct 26 03:33 ..
lrwxrwxrwx 1 root root   54 Oct 26 03:33 ocr -> /content/drive/MyDrive/cv_experience_ocr_bert/data/ocr
drwxr-xr-x 2 root root 4096 Oct 26 03:33 processed
lrwxrwxrwx 1 root root   54 Oct 26 03:33 raw -> /content/drive/MyDrive/cv_experience_ocr_bert/data/raw
drwxr-xr-x 2 root root 4096 Oct 26 03:33 splits

models:
total 12
drwxr-xr-x 3 root root 4096 Oct 26 03:33 .
drwxr-xr-x 9 root root 4096 Oct 26 03:33 ..
drwxr-xr-x 2 root root 4096 Oct 26 03:33 baselines
lrwxrwxrwx 1 root root   57 Oct 26 03:33 bert -> /content/drive/MyDrive/cv_experience_ocr_bert/models/bert


In [3]:
# Dependencias (ver requirements-colab.txt). Se congelan en la versión que trae
# Colab solo las librerías de las que dependen sus herramientas internas:
#   pandas -> google-colab exige su versión exacta
#   numpy  -> numba y buena parte del stack de Colab
#   torch  -> build con CUDA de Colab
# pyarrow NO se congela: datasets 5.1 exige pyarrow>=24 y lo único en Colab que
# pide <24 es cudf, que este proyecto no usa.
# Si algo exigiera otra versión de las congeladas, pip falla aquí de forma
# explícita en vez de romper Colab en silencio.
import numpy, pandas, torch
keep = [f'{m.__name__}=={m.__version__.split("+")[0]}' for m in (numpy, pandas, torch)]
with open('/tmp/colab_keep.txt', 'w') as f:
    f.write('\n'.join(keep))
print('Se conservan:', keep)
!pip install -q -r requirements-colab.txt -c /tmp/colab_keep.txt

Se conservan: ['numpy==2.1.3', 'pandas==2.2.3', 'torch==2.11.0']
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 2.3.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


## 1. Descarga de datos (una sola vez)

Muestrea 800 CVs sin foto por ocupación (semilla 42) y guarda las imágenes en Drive. Toma del orden de 15–30 minutos. Si se corta, vuelve a correrla: el muestreo es determinista y produce la misma muestra.

In [6]:
!python -m src.download_data

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/cv_experience_ocr_bert/src/download_data.py", line 117, in <module>
    main(args.n_per_occupation, args.occupations, args.shuffle_buffer)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/cv_experience_ocr_bert/src/download_data.py", line 85, in main
    rows += sample_occupation(job, n_per_occupation, token, buffer_size)
            ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/cv_experience_ocr_bert/src/download_data.py", line 44, in sample_occupation
    ds = load_dataset(
        HF_DATASET,
    ...<4 lines>...
        token=token,
    )
  File "/usr/local/lib/python3.13/dist-packages/datasets/load.py", line 1667, in load_dataset
    builder_instance = load_dataset_builder(
        path=path,
    ...<10 lines>...
        **config_kwargs,
    )
  File "/usr/l

In [ ]:
# sampled_ids.csv vive en el repo (se versiona). Se copia a Drive para que
# lo bajes a la laptop y lo subas en un commit.
!cp data/sampled_ids.csv {DRIVE}/data/sampled_ids.csv
!ls {DRIVE}/data/raw | head; ls {DRIVE}/data/raw/images | wc -l